# Section 4.5 — Corrupted Initialization: Person Counting

Three arms, same data, same tuning. Only the initial parameters and
whether learning is switched on differ.

| Arm | init | learning |
|---|---|---|
| Naive-frozen | b = 0, c = 1, σ² = 1 | off |
| Sabotaged-frozen | b = 0, c = −3, σ² = 3 | off |
| AKF-recovery | b = 0, c = −3, σ² = 3 | on from k = 0 |

The counting task uses the affine measurement model y = b + c·X + v, with the
fused state augmented to [X, 1] so that each microphone's offset b is
estimated alongside its coupling c. The filter in Section 2 is the same one
that produced the reported counting results (12.34 / 38.56 / 49.70 % for the
annotated / expert / fused teachers).

There is no burn-in: the estimator is allowed to update from the first
sample. While the teacher has visited only one class level the affine fit is
rank-deficient, and `fit_affine_sensor` keeps the previous (b, c) until a
second level appears.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
sns.set_palette("deep")
np.random.seed(1)

# use the exact filename shown by os.listdir("/content")
CSV = "../data/person_counting_2163.csv"

# shared tuning, identical across all three arms
sig1, c1 = 1.0, 1.0
Q_val, P0_val, X0_val = 10.0, 1.0, 0.0

CLASS_LEVELS = np.array([0, 1, 2])

# ---- the bad start ----
# An inverted coupling. With b = 0 and c = -3 the microphones push the
# fused estimate in the opposite direction to the true count.
SABOTAGE_B = 0.0
SABOTAGE_C = -3.0
SABOTAGE_S = 3.0

TEACHER_MODE = "gt"      # "gt", "expert", or "fused"


## 1. Load and build ground truth

Half-open interval fills from paired marker events (Chapter 3).


In [2]:
df = pd.read_csv("../data/person_counting_2163.csv")
df = df[["human_count", "mouse_event", "dBFS_mic1", "dBFS_mic2"]].dropna()
df = df.rename(columns={"human_count": "human", "mouse_event": "mouse"})
df = df.reset_index(drop=True)

m = df["mouse"].astype(int).to_numpy()
count_gt = np.zeros(len(m), dtype=int)
active = 0
for k, event in enumerate(m):
    if event == 1:
        active = 0 if active == 1 else 1
    elif event == 2:
        active = 0 if active == 2 else 2
    count_gt[k] = active

Np = len(df)
N  = Np
Xd = count_gt
X  = Xd                                              # alias used below
y1 = df["human"].astype(float).to_numpy()            # expert (camera count)
y0 = df["dBFS_mic1"].astype(float).to_numpy()        # NOT centred: b is learned
y2 = df["dBFS_mic2"].astype(float).to_numpy()

counts = np.bincount(Xd, minlength=3).tolist()
print(f"N = {N}   class counts 0/1/2 = {counts}")
assert N == 2163 and counts == [860, 559, 744], "does not match Table 1"

N = 2163   class counts 0/1/2 = [860, 559, 744]


## 2. The filter

Unchanged from the verified counting bad-start script. Parameter histories
are recorded after the update at each k, so the initial value is prepended
when plotting.


In [3]:
def quantize_count(x):
    return int(CLASS_LEVELS[np.argmin(np.abs(CLASS_LEVELS - float(x)))])


def fit_affine_sensor(y, x, prev_b, prev_c):
    """Least-squares fit of y ~= b + c*x. Falls back to the previous
    estimate while the teacher has visited fewer than two distinct levels."""
    x = np.asarray(x, dtype=float).reshape(-1)
    y = np.asarray(y, dtype=float).reshape(-1)
    H = np.column_stack([x, np.ones_like(x)])
    if np.linalg.matrix_rank(H) < 2:
        resid = y - (prev_b + prev_c * x)
        return prev_b, prev_c, max(float(np.var(resid)), 1e-6)
    theta = np.linalg.pinv(H) @ y
    c_hat, b_hat = float(theta[0]), float(theta[1])
    resid = y - (b_hat + c_hat * x)
    return b_hat, c_hat, max(float(np.var(resid)), 1e-6)


def run_counting_kf(b_init, c_init, r_init, learn, teacher_mode="gt"):
    """Expert + two-inexpert affine AKF for the counting task.

    learn=False freezes (b, c, sigma^2) at their initial values throughout.
    learn=True  updates them online from k = 0 (no burn-in).
    """
    # --- expert-only KF (scalar state) ---
    X_1 = np.array([X0_val])
    P_1 = np.eye(1) * P0_val
    Q_1 = Q_val * np.eye(1)
    R_1 = np.array([[sig1 ** 2]])
    C_1 = np.array([[c1]])
    A_1 = np.eye(1)

    # --- fused KF (augmented state [X, 1]) ---
    X_all = np.array([X0_val, 1.0])
    P_all = np.diag([P0_val, 0.0])
    Q_all = np.diag([Q_val, 0.0])
    A_all = np.eye(2)

    bh0, ch0, rh0 = float(b_init), float(c_init), float(r_init)
    bh2, ch2, rh2 = float(b_init), float(c_init), float(r_init)

    expert_cls = np.zeros(Np, dtype=int)
    fused_cls  = np.zeros(Np, dtype=int)
    hist = {k: np.zeros(Np) for k in
            ("bh0", "ch0", "rh0", "bh2", "ch2", "rh2")}

    for k in range(Np):

        # --- expert-only update ---
        Xm = A_1 @ X_1
        Pm = A_1 @ P_1 @ A_1.T + Q_1
        res = y1[k] - (C_1 @ Xm)
        K = Pm @ C_1.T @ np.linalg.inv(C_1 @ Pm @ C_1.T + R_1)
        X_1 = (Xm + K @ res).flatten()
        P_1 = (np.eye(1) - K @ C_1) @ Pm
        expert_cls[k] = quantize_count(X_1[0])

        # --- fused update ---
        rh0 = max(float(rh0), 1e-6)
        rh2 = max(float(rh2), 1e-6)
        R_all = np.diag([sig1 ** 2, rh0, rh2])
        C_all = np.array([[c1, 0.0], [ch0, bh0], [ch2, bh2]])

        Xm = A_all @ X_all.reshape(-1, 1)
        Pm = A_all @ P_all @ A_all.T + Q_all
        y_vec = np.array([y1[k], y0[k], y2[k]]).reshape(-1, 1)
        res_v = y_vec - C_all @ Xm
        K = Pm @ C_all.T @ np.linalg.inv(C_all @ Pm @ C_all.T + R_all)
        X_all = (Xm + K @ res_v).flatten()
        P_all = (np.eye(2) - K @ C_all) @ Pm
        X_all[1] = 1.0                      # hold the constant state
        fused_cls[k] = quantize_count(X_all[0])

        # --- parameter learning (from k = 0) ---
        if learn:
            if teacher_mode == "gt":
                teacher = Xd[:k + 1].astype(float)
            elif teacher_mode == "expert":
                teacher = expert_cls[:k + 1].astype(float)
            elif teacher_mode == "fused":
                teacher = fused_cls[:k + 1].astype(float)
            else:
                raise ValueError(f"unknown teacher_mode {teacher_mode!r}")

            bh0, ch0, rh0 = fit_affine_sensor(y0[:k + 1], teacher, bh0, ch0)
            bh2, ch2, rh2 = fit_affine_sensor(y2[:k + 1], teacher, bh2, ch2)

        for name, val in (("bh0", bh0), ("ch0", ch0), ("rh0", rh0),
                          ("bh2", bh2), ("ch2", ch2), ("rh2", rh2)):
            hist[name][k] = val

    return expert_cls, fused_cls, hist


def cum_err(dec):
    return 100 * np.cumsum(dec != X) / (np.arange(N) + 1)


def confusion(y_true, y_pred):
    cm = np.zeros((3, 3), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def recalls(cm):
    return [100 * cm[i, i] / cm[i].sum() for i in range(3)]


### Offline reference

Full-record least-squares fit of each microphone against the annotations.
This is the value the annotated teacher reaches at k = N−1; it is shown on
the parameter plots as a reference line, not as a true calibration value.


In [4]:
def offline_ls(y):
    H = np.column_stack([Xd.astype(float), np.ones(N)])
    c_, b_ = np.linalg.pinv(H) @ y
    return float(b_), float(c_)

B_REF, C_REF = {}, {}
B_REF[1], C_REF[1] = offline_ls(y0)
B_REF[2], C_REF[2] = offline_ls(y2)
for i in (1, 2):
    print(f"mic {i}:  b* = {B_REF[i]:9.4f}   c* = {C_REF[i]:7.4f}")


mic 1:  b* =  -39.8997   c* =  7.0070
mic 2:  b* =  -35.3774   c* =  7.1371


## 3. Run the three arms


In [5]:
_, naive_dec, naive_h = run_counting_kf(0.0, 1.0, 1.0, learn=False)
_, sab_dec,   sab_h   = run_counting_kf(SABOTAGE_B, SABOTAGE_C, SABOTAGE_S, learn=False)
expert_dec, akf_dec, akf_h = run_counting_kf(SABOTAGE_B, SABOTAGE_C, SABOTAGE_S,
                                             learn=True, teacher_mode=TEACHER_MODE)

cum_naive, cum_sab, cum_akf = cum_err(naive_dec), cum_err(sab_dec), cum_err(akf_dec)
cum_expert = cum_err(expert_dec)

print(f"teacher = {TEACHER_MODE}   learning from k = 0 (no burn-in)\n")
print(f"  Expert-only KF                                  : {cum_expert[-1]:6.2f}%")
print(f"  Naive-frozen     (b=0, c=1, s=1, frozen)        : {cum_naive[-1]:6.2f}%")
print(f"  Sabotaged-frozen (b={SABOTAGE_B:.0f}, c={SABOTAGE_C:.0f}, s={SABOTAGE_S:.0f}, frozen)      : {cum_sab[-1]:6.2f}%")
print(f"  AKF-recovery     (same init, learning on)       : {cum_akf[-1]:6.2f}%")

print("\nPer-class recall [%]   class 0   class 1   class 2")
for label, dec in (("Naive-frozen", naive_dec), ("Sabotaged-frozen", sab_dec),
                   ("AKF-recovery", akf_dec)):
    r = recalls(confusion(X, dec))
    print(f"  {label:<18}  {r[0]:7.2f}   {r[1]:7.2f}   {r[2]:7.2f}")

print("\nPredicted class counts (0/1/2):")
for label, dec in (("Naive-frozen", naive_dec), ("Sabotaged-frozen", sab_dec),
                   ("AKF-recovery", akf_dec)):
    print(f"  {label:<18}  {np.bincount(dec, minlength=3).tolist()}")

print(f"\nAKF final parameters:")
print(f"  b1 = {akf_h['bh0'][-1]:9.4f}  c1 = {akf_h['ch0'][-1]:7.4f}  (ref b*={B_REF[1]:.4f}, c*={C_REF[1]:.4f})")
print(f"  b2 = {akf_h['bh2'][-1]:9.4f}  c2 = {akf_h['ch2'][-1]:7.4f}  (ref b*={B_REF[2]:.4f}, c*={C_REF[2]:.4f})")
print(f"\nAKF confusion (rows true 0/1/2):\n{confusion(X, akf_dec)}")


teacher = gt   learning from k = 0 (no burn-in)

  Expert-only KF                                  :  44.24%
  Naive-frozen     (b=0, c=1, s=1, frozen)        :  60.24%
  Sabotaged-frozen (b=0, c=-3, s=3, frozen)      :  65.60%
  AKF-recovery     (same init, learning on)       :  13.04%

Per-class recall [%]   class 0   class 1   class 2
  Naive-frozen         100.00      0.00      0.00
  Sabotaged-frozen       0.00      0.00    100.00
  AKF-recovery          97.09     86.76     75.40

Predicted class counts (0/1/2):
  Naive-frozen        [2163, 0, 0]
  Sabotaged-frozen    [0, 0, 2163]
  AKF-recovery        [864, 671, 628]

AKF final parameters:
  b1 =  -39.8997  c1 =  7.0070  (ref b*=-39.8997, c*=7.0070)
  b2 =  -35.3774  c2 =  7.1371  (ref b*=-35.3774, c*=7.1371)

AKF confusion (rows true 0/1/2):
[[835   8  17]
 [ 24 485  50]
 [  5 178 561]]


## 4. Does the learning actually do anything?

The rescue rate restricts attention to the samples where the camera's own
count is wrong — the only regime in which microphone calibration can change
the outcome.


In [6]:
camera_cls = np.clip(np.rint(y1), 0, 2).astype(int)
wrong = (camera_cls != X)
n_wrong = int(wrong.sum())

print(f"camera wrong on {n_wrong}/{N} samples ({100*n_wrong/N:.2f}%)\n")
for label, dec in (("Naive-frozen", naive_dec),
                   ("Sabotaged-frozen", sab_dec),
                   ("AKF-recovery", akf_dec)):
    r = 100 * ((dec == X) & wrong).sum() / n_wrong
    print(f"  {label:<18} rescue rate {r:6.2f}%")

n_diff = int((naive_dec != akf_dec).sum())
print(f"\nnaive vs AKF differ at {n_diff}/{N} samples")


camera wrong on 957/2163 samples (44.24%)

  Naive-frozen       rescue rate   0.00%
  Sabotaged-frozen   rescue rate  59.77%
  AKF-recovery       rescue rate  76.38%

naive vs AKF differ at 1299/2163 samples


## 5. Recovery point

The first index after which the AKF error stays permanently below half the
sabotaged-frozen final error, and the first index after which each learned
coupling stays within 10 % of its final value.


In [7]:
thr = 0.5 * cum_sab[-1]
below = cum_akf <= thr
rec = next((i for i in range(N) if below[i] and below[i:].all()), None)

if rec is not None:
    print(f"AKF error drops permanently below {thr:.2f}% at k = {rec}")
else:
    print(f"AKF error never settles permanently below {thr:.2f}%")

gap = cum_sab - cum_akf
g = int(np.argmax(gap))
print(f"largest gap: {gap[g]:.2f} points at k = {g} "
      f"(AKF {cum_akf[g]:.2f}%, sabotaged {cum_sab[g]:.2f}%)")


def settle(h, tol=0.10):
    f = h[-1]
    if abs(f) < 1e-9:
        return None
    ok = np.abs(h - f) <= tol * abs(f)
    return next((i for i in range(len(ok)) if ok[i] and ok[i:].all()), None)

print(f"\nc1 within 10% of final from k = {settle(akf_h['ch0'])}")
print(f"c2 within 10% of final from k = {settle(akf_h['ch2'])}")

# first sample at which the teacher has visited two class levels,
# i.e. the first k at which the affine fit can move away from the start
T_used = {"gt": X, "expert": expert_dec, "fused": akf_dec}[TEACHER_MODE]
first_two = next(k for k in range(N) if len(np.unique(T_used[:k + 1])) >= 2)
print(f"teacher first spans two class levels at k = {first_two}")


AKF error drops permanently below 32.80% at k = 170
largest gap: 72.15 points at k = 78 (AKF 27.85%, sabotaged 100.00%)

c1 within 10% of final from k = 196
c2 within 10% of final from k = 561
teacher first spans two class levels at k = 15


## 6. Figures


In [8]:
kk = np.arange(-1, N)    # prepend the initial value at k = -1

# ---- Figure: the three curves ----
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(cum_naive, ls="--", color="gray", lw=1.6,
        label=f"Naive-frozen ({cum_naive[-1]:.2f}%)")
ax.plot(cum_sab, ls=":", color="firebrick", lw=1.6,
        label=f"Sabotaged-frozen ({cum_sab[-1]:.2f}%)")
ax.plot(cum_akf, color="steelblue", lw=2,
        label=f"AKF-recovery ({cum_akf[-1]:.2f}%)")
ax.set_xlabel("Time step $k$")
ax.set_ylabel("Cumulative classification error [%]")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("badstart_error_multiclass.pdf")
plt.show()

# ---- Figure: coupling recovery ----
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for idx, (ax, key) in enumerate(zip(axes, ("ch0", "ch2")), start=1):
    ax.axhline(C_REF[idx], color="black", ls="--", lw=1.4,
               label=rf"Offline reference $c_{idx}^*$ = {C_REF[idx]:.3f}")
    ax.axhline(SABOTAGE_C, color="firebrick", ls=":", lw=1.4,
               label=f"Sabotaged start ({SABOTAGE_C:.0f})")
    ax.plot(kk, np.r_[SABOTAGE_C, akf_h[key]], color="steelblue", lw=1.8,
            label=f"AKF-recovery ({akf_h[key][-1]:.3f})")
    ax.set_ylabel(rf"$\hat{{c}}_{idx}(k)$")
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right", fontsize=8)
axes[1].set_xlabel("Time step $k$")
fig.tight_layout()
fig.savefig("badstart_couplings_multiclass.pdf")
plt.show()

# ---- Figure: offset recovery ----
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for idx, (ax, key) in enumerate(zip(axes, ("bh0", "bh2")), start=1):
    ax.axhline(B_REF[idx], color="black", ls="--", lw=1.4,
               label=rf"Offline reference $b_{idx}^*$ = {B_REF[idx]:.3f}")
    ax.axhline(SABOTAGE_B, color="firebrick", ls=":", lw=1.4,
               label=f"Sabotaged start ({SABOTAGE_B:.0f})")
    ax.plot(kk, np.r_[SABOTAGE_B, akf_h[key]], color="steelblue", lw=1.8,
            label=f"AKF-recovery ({akf_h[key][-1]:.3f})")
    ax.set_ylabel(rf"$\hat{{b}}_{idx}(k)$ [dBFS]")
    ax.grid(alpha=0.3)
    ax.legend(loc="center right", fontsize=8)
axes[1].set_xlabel("Time step $k$")
fig.tight_layout()
fig.savefig("badstart_offsets_multiclass.pdf")
plt.show()

print("saved: badstart_error_multiclass.pdf, badstart_couplings_multiclass.pdf, "
      "badstart_offsets_multiclass.pdf")


/tmp/ipykernel_1108/3940854379.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_1108/3940854379.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved: badstart_error_multiclass.pdf, badstart_couplings_multiclass.pdf, badstart_offsets_multiclass.pdf


/tmp/ipykernel_1108/3940854379.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Annotated vs self-taught from the inverted start

Same corruption, same data; only the teaching signal differs.


In [9]:
INVERT_B, INVERT_C, INVERT_S = 0.0, -3.0, 3.0

_, gt_dec,  gt_h  = run_counting_kf(INVERT_B, INVERT_C, INVERT_S, learn=True, teacher_mode="gt")
_, fus_dec, fus_h = run_counting_kf(INVERT_B, INVERT_C, INVERT_S, learn=True, teacher_mode="fused")

for label, dec, h in (("annotated  ", gt_dec, gt_h), ("self-taught", fus_dec, fus_h)):
    r = recalls(confusion(X, dec))
    print(f"{label} final error {100*(dec != X).mean():6.2f}%"
          f"   c1 {h['ch0'][-1]:7.3f}  c2 {h['ch2'][-1]:7.3f}"
          f"   recall 0/1/2 {r[0]:.1f}/{r[1]:.1f}/{r[2]:.1f}")

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
for idx, (ax, key) in enumerate(zip(axes, ("ch0", "ch2")), start=1):
    ax.axhline(0, color="gray", lw=0.8, alpha=0.6)
    ax.axhline(C_REF[idx], color="black", ls="--", lw=1.4,
               label=rf"Offline reference $c_{idx}^*$ = {C_REF[idx]:.3f}")
    ax.axhline(INVERT_C, color="firebrick", ls=":", lw=1.4,
               label=f"Inverted start ({INVERT_C:.0f})")
    ax.plot(kk, np.r_[INVERT_C, gt_h[key]], color="seagreen", lw=1.8,
            label=f"Annotated ({gt_h[key][-1]:.3f})")
    ax.plot(kk, np.r_[INVERT_C, fus_h[key]], color="steelblue", lw=1.8,
            label=f"Self-taught ({fus_h[key][-1]:.3f})")
    ax.set_ylabel(rf"$\hat{{c}}_{idx}(k)$")
    ax.grid(alpha=0.3)
    ax.legend(loc="center right", fontsize=8)
axes[1].set_xlabel("Time step $k$")
fig.tight_layout()
fig.savefig("corrupted_couplings_multiclass.pdf")
plt.show()


annotated   final error  13.04%   c1   7.007  c2   7.137   recall 0/1/2 97.1/86.8/75.4
self-taught final error  84.56%   c1  -6.076  c2  -4.955   recall 0/1/2 1.0/42.4/11.8


/tmp/ipykernel_1108/1903470234.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Summary table (all teachers, inverted start)

One row per teacher, for the results table in the paper.


In [10]:
rows = []
for t in ("gt", "expert", "fused"):
    _, d, h = run_counting_kf(SABOTAGE_B, SABOTAGE_C, SABOTAGE_S,
                              learn=True, teacher_mode=t)
    r = recalls(confusion(X, d))
    rows.append([t, cum_sab[-1], 100 * (d != X).mean(), r[0], r[1], r[2],
                 h["bh0"][-1], h["ch0"][-1], h["bh2"][-1], h["ch2"][-1]])

summary = pd.DataFrame(rows, columns=["teacher", "sab-frozen %", "AKF %",
                                      "rec0 %", "rec1 %", "rec2 %",
                                      "b1", "c1", "b2", "c2"])
print(f"naive-frozen: {cum_naive[-1]:.2f}%   expert-only: {cum_expert[-1]:.2f}%\n")
print(summary.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


naive-frozen: 60.24%   expert-only: 44.24%

teacher  sab-frozen %   AKF %  rec0 %  rec1 %  rec2 %       b1      c1       b2      c2
     gt       65.6033 13.0374 97.0930 86.7621 75.4032 -39.8997  7.0070 -35.3774  7.1371
 expert       65.6033 39.2510 97.9070 46.1538 28.7634 -34.5188  4.0126 -29.7472  3.6074
  fused       65.6033 84.5585  1.0465 42.3971 11.8280 -28.4510 -6.0759 -24.6946 -4.9547
